# Seed Ensemble — In-Domain Five-Class Accuracy (no training)

Each leave-one-out fold was trained three times with different random seeds. This notebook averages the three seeds' predicted probabilities per fold and recomputes in-domain (validation) five-class accuracy. Averaging independent seeds typically gives a small, genuine accuracy gain at no compute cost, because it cancels some of the per-seed variance.

No training is performed. The notebook reads the saved per-seed prediction files and combines them.

## Setup

Locates the saved per-seed prediction files. The pipeline saved one file per fold and seed; these contain the true labels and the predicted five-class probabilities for the validation (in-domain) set.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, glob, os, json, re
from pathlib import Path
from sklearn.metrics import accuracy_score, cohen_kappa_score

# Adjust if your results live elsewhere.
SEARCH_DIRS = [
    '/content/drive/MyDrive/Master Thesis/scope3/results',
    '/content/drive/MyDrive/Master Thesis/scope3_mt/results',
]
all_files = []
for d in SEARCH_DIRS:
    all_files += glob.glob(os.path.join(d, '*preds*.npz'))
all_files = sorted(set(all_files))
print('Found prediction files:')
for f in all_files: print('  ', os.path.basename(f))

Mounted at /content/drive
Found prediction files:
   ablation_A_naive_seed0_preds.npz
   ablation_B_sampler_seed0_preds.npz
   ablation_C_noiseloss_seed0_preds.npz
   ablation_D_curriculum_seed0_preds.npz
   ablation_E_domain_adv_seed0_preds.npz
   ablation_F_direct5class_seed0_preds.npz
   ablation_G_ordinal_seed0_preds.npz
   ablation_H_combined_seed0_preds.npz
   fold1_test_mendeley_seed0_preds.npz
   fold1_test_mendeley_seed1_preds.npz
   fold1_test_mendeley_seed2_preds.npz
   fold2_test_mrkr_seed0_preds.npz
   fold2_test_mrkr_seed1_preds.npz
   fold2_test_mrkr_seed2_preds.npz
   fold3_test_nhanes3_seed0_preds.npz
   fold3_test_nhanes3_seed1_preds.npz
   fold3_test_nhanes3_seed2_preds.npz
   fold4_test_oai_seed0_preds.npz
   fold4_test_oai_seed1_preds.npz
   fold4_test_oai_seed2_preds.npz
   grlsweep_lambda0.1_seed0_preds.npz
   grlsweep_lambda0.3_seed0_preds.npz
   grlsweep_lambda0.5_seed0_preds.npz
   quality_estimator_seed0_preds.npz
   single_mrkr_seed0_preds.npz
   single_nhan

## Identify the in-domain (validation) prediction arrays

The averaging requires per-seed probabilities on the same validation set. The cell below inspects each file's contents and keeps those that contain internal/validation probabilities. If the saved files store only external predictions, the notebook reports that and the ensemble cannot be formed without re-running validation inference; in that case use the surgical inference notebook to regenerate per-seed validation probabilities first.

In [ ]:
def inspect(f):
    d = np.load(f)
    return list(d.files)

print('File contents:')
for f in all_files:
    keys = inspect(f)
    print(f'  {os.path.basename(f):45s} -> {keys}')

# Heuristic: group files by fold (dataset) and seed from the filename.
def parse(fn):
    base = os.path.basename(fn)
    ds = None
    for cand in ['mendeley','mrkr','nhanes3','oai']:
        if cand in base: ds = cand; break
    m = re.search(r'seed(\d+)', base)
    seed = int(m.group(1)) if m else None
    # internal vs external hint
    kind = 'internal' if ('val' in base or 'internal' in base) else 'unknown'
    return ds, seed, kind

groups = {}
for f in all_files:
    ds, seed, kind = parse(f)
    if ds is None or seed is None: continue
    groups.setdefault(ds, {})[seed] = f
print()
print('Folds with multiple seeds available:')
for ds, sm in groups.items():
    print(f'  {ds}: seeds {sorted(sm.keys())}')

File contents:
  ablation_A_naive_seed0_preds.npz              -> ['y_true', 'y_pred', 'probs']
  ablation_B_sampler_seed0_preds.npz            -> ['y_true', 'y_pred', 'probs']
  ablation_C_noiseloss_seed0_preds.npz          -> ['y_true', 'y_pred', 'probs']
  ablation_D_curriculum_seed0_preds.npz         -> ['y_true', 'y_pred', 'probs']
  ablation_E_domain_adv_seed0_preds.npz         -> ['y_true', 'y_pred', 'probs']
  ablation_F_direct5class_seed0_preds.npz       -> ['y_true', 'y_pred', 'probs']
  ablation_G_ordinal_seed0_preds.npz            -> ['y_true', 'y_pred', 'probs']
  ablation_H_combined_seed0_preds.npz           -> ['y_true', 'y_pred', 'probs']
  fold1_test_mendeley_seed0_preds.npz           -> ['y_true', 'y_pred', 'probs']
  fold1_test_mendeley_seed1_preds.npz           -> ['y_true', 'y_pred', 'probs']
  fold1_test_mendeley_seed2_preds.npz           -> ['y_true', 'y_pred', 'probs']
  fold2_test_mrkr_seed0_preds.npz               -> ['y_true', 'y_pred', 'probs']
  fold2_test_

## Form the ensemble and score

For each fold, the per-seed probability arrays are averaged and the argmax gives the ensembled prediction. Accuracy and quadratic weighted kappa are computed against the true labels. The single-seed mean is shown alongside, so the ensemble gain is visible directly.

The averaging assumes the three seed files store probabilities for the same set of images in the same order. If a file stores only hard predictions (no probabilities), majority vote across seeds is used instead.

In [ ]:
PROB_KEY_CANDIDATES = ['probs','prob','val_probs','internal_probs','y_prob']
PRED_KEY_CANDIDATES = ['y_pred','pred','val_pred']
TRUE_KEY_CANDIDATES = ['y_true','y','val_true','labels']

def pick(d, cands):
    for k in cands:
        if k in d.files: return d[k]
    return None

results = {}
for ds, seedmap in groups.items():
    seeds = sorted(seedmap.keys())
    if len(seeds) < 2:
        continue
    prob_stack = []; preds_stack = []; ytrue = None; ok = True
    for s in seeds:
        d = np.load(seedmap[s])
        yt = pick(d, TRUE_KEY_CANDIDATES)
        if yt is None: ok = False; break
        if ytrue is None: ytrue = yt
        pr = pick(d, PROB_KEY_CANDIDATES)
        if pr is not None:
            prob_stack.append(pr)
        else:
            yp = pick(d, PRED_KEY_CANDIDATES)
            if yp is None: ok = False; break
            preds_stack.append(yp)
    if not ok or ytrue is None:
        print(f'{ds}: could not assemble (missing keys), skipped')
        continue

    # single-seed mean accuracy for reference
    single_acc = []
    for s in seeds:
        d = np.load(seedmap[s]); yt = pick(d, TRUE_KEY_CANDIDATES)
        yp = pick(d, PRED_KEY_CANDIDATES)
        if yp is None and pick(d, PROB_KEY_CANDIDATES) is not None:
            yp = pick(d, PROB_KEY_CANDIDATES).argmax(1)
        single_acc.append(accuracy_score(yt, yp))

    if prob_stack:
        avg = np.mean(np.stack(prob_stack, 0), axis=0)
        ens_pred = avg.argmax(1)
        mode = 'probability average'
    else:
        P = np.stack(preds_stack, 0)
        ens_pred = np.apply_along_axis(lambda col: np.bincount(col).argmax(), 0, P)
        mode = 'majority vote'

    ens_acc = accuracy_score(ytrue, ens_pred)
    ens_qwk = cohen_kappa_score(ytrue, ens_pred, weights='quadratic')
    results[ds] = {'single_mean': float(np.mean(single_acc)), 'ensemble_acc': float(ens_acc),
                   'ensemble_qwk': float(ens_qwk), 'gain': float(ens_acc-np.mean(single_acc)), 'mode': mode, 'seeds': seeds}
    print(f'{ds}: {mode}, seeds {seeds} -> single {np.mean(single_acc):.3f}, ensemble {ens_acc:.3f} ({ens_acc-np.mean(single_acc):+.3f})')
print('Done.')

mendeley: probability average, seeds [0, 1, 2] -> single 0.373, ensemble 0.393 (+0.021)
mrkr: probability average, seeds [0, 1, 2] -> single 0.495, ensemble 0.527 (+0.032)
nhanes3: probability average, seeds [0, 1, 2] -> single 0.594, ensemble 0.621 (+0.027)
oai: probability average, seeds [0, 1, 2] -> single 0.503, ensemble 0.522 (+0.019)
Done.


## Results table

In-domain five-class accuracy: single-seed mean versus the seed ensemble, per fold, with the mean across folds.

In [ ]:
if results:
    print('='*66)
    print('IN-DOMAIN FIVE-CLASS ACCURACY — single seed vs seed ensemble')
    print('='*66)
    print(f"{'Fold':12s}{'Single mean':>14s}{'Ensemble':>12s}{'Gain':>10s}{'Ens. QWK':>11s}")
    print('-'*66)
    sm=[]; en=[]
    for ds, r in results.items():
        print(f"{ds:12s}{r['single_mean']:>14.3f}{r['ensemble_acc']:>12.3f}{r['gain']:>+10.3f}{r['ensemble_qwk']:>11.3f}")
        sm.append(r['single_mean']); en.append(r['ensemble_acc'])
    print('-'*66)
    print(f"{'MEAN':12s}{np.mean(sm):>14.3f}{np.mean(en):>12.3f}{np.mean(en)-np.mean(sm):>+10.3f}")
    print('='*66)
    json.dump(results, open('/content/drive/MyDrive/Master Thesis/scope3_mt/results/seed_ensemble.json','w'), indent=2)
    print('Saved -> seed_ensemble.json')
else:
    print('No folds had the required per-seed probability files.')
    print('If the saved files contain only external predictions, regenerate per-seed')
    print('validation probabilities with the inference notebook, then re-run this.')

IN-DOMAIN FIVE-CLASS ACCURACY — single seed vs seed ensemble
Fold           Single mean    Ensemble      Gain   Ens. QWK
------------------------------------------------------------------
mendeley             0.373       0.393    +0.021      0.435
mrkr                 0.495       0.527    +0.032      0.598
nhanes3              0.594       0.621    +0.027      0.637
oai                  0.503       0.522    +0.019      0.592
------------------------------------------------------------------
MEAN                 0.491       0.516    +0.025
Saved -> seed_ensemble.json
